## 02_xi_cw — Coles × Woolworths 交差相関関数 ξ_CW(r)

**入力**
- `output/data/coles_locations.csv`      — Coles 店舗座標 (N_C = 685)
- `output/data/woolworths_locations.csv` — Woolworths 店舗座標 (N_W = 1,039)
- `output/data/random_masked.csv`        — 共有大陸マスク済みランダムカタログ (N_R = 10,390)
- `output/xi/xi_cc.csv`                  — ξ_CC（ビン設計確認用）

**出力**
- `output/xi/xi_cw_wide.csv`   — ξ_CW(r): 対数ビン, 全スケール (38 ビン)
- `output/xi/xi_cw_small.csv`  — ξ_CW(r): 線形ビン Δr=1 km, r=0.5–30.5 km

**前提**
- `00_random_catalog.ipynb`, `01_xi_cc.ipynb`, `01_xi_ww.ipynb` 実行済み
- `gradle :lib:jar` 完了

**推定量**
$$\hat{\xi}_{CW}(r) = \frac{D_C D_W - D_C R - D_W R + R^2}{R^2}$$

共有ランダムカタログ $R$ を使用するため $D_C R$ と $D_W R$ は同一 $R$ に対する計算。

In [1]:
@file:DependsOn("../../lib/build/libs/retail-utils-1.0.jar")

In [2]:
%use dataframe
%use lets-plot

import retail.*
import kotlin.math.*

In [3]:
// --- データ読み込み ---
val dfC      = DataFrame.readCSV("./output/data/coles_locations.csv")
val dfW      = DataFrame.readCSV("./output/data/woolworths_locations.csv")
val dfRandom = DataFrame.readCSV("./output/data/random_masked.csv")

val colesPoints: List<Point>  = dfC.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }
val woolPoints:  List<Point>  = dfW.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }
val randPoints:  List<Point>  = dfRandom.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }

val nC = colesPoints.size;  val nW = woolPoints.size;  val nR = randPoints.size
println("N_C = $nC  |  N_W = $nW  |  N_R = $nR")

N_C = 685  |  N_W = 1039  |  N_R = 10390


In [4]:
// --- ビン設計（xi_cc.csv から rMin を取得して Coles と完全一致）---
val xiCcDf = DataFrame.readCSV("./output/xi/xi_cc.csv")
val rMinFromCC = xiCcDf.rows().first()["r_km"] as Double * 0.95  // ビン端を逆算
// より直接的に: Coles の meanNN を再計算
val meanNNc = colesPoints.map { p -> colesPoints.filter { it !== p }.minOf { haversine(p, it) } }.average()
val bins     = logBins(rMin = meanNNc / 2.0)
val centers  = binCenters(bins)
val nBins    = centers.size
println("rMin = %.2f km  |  nBins = $nBins".format(meanNNc / 2.0))

rMin = 7.63 km  |  nBins = 38


In [5]:
// --- 交差ペアカウント ---
// D_C × D_W: 異種間全ペア（i < j 除外なし）
// D_C × R, D_W × R: 各データと共有ランダムカタログのペア
// R × R: ランダム自己ペア

println("D_C × D_W を計算中...")
val nDcDw = pairCounts(colesPoints, woolPoints, bins)

println("D_C × R を計算中...")
val nDcR = pairCounts(colesPoints, randPoints, bins)

println("D_W × R を計算中...")
val nDwR = pairCounts(woolPoints, randPoints, bins)

println("R × R を計算中...")
val nRR  = pairCounts(randPoints, null, bins)

println("完了")

D_C × D_W を計算中...
D_C × R を計算中...
D_W × R を計算中...
R × R を計算中...
完了


In [6]:
// --- ξ_CW 推定量 ---
// ξ_CW = (D_C D_W - D_C R - D_W R + R^2) / R^2
// 正規化: DC×DW → N_C×N_W, DCR → N_C×N_R, DWR → N_W×N_R, RR → N_R×(N_R-1)/2
val normDcDw = nC.toLong() * nW
val normDcR  = nC.toLong() * nR
val normDwR  = nW.toLong() * nR
val normRR   = nR.toLong() * (nR - 1) / 2

val xiCW: List<XiBin> = centers.indices.map { i ->
    val dcDw = nDcDw[i].toDouble() / normDcDw
    val dcR  = nDcR[i].toDouble()  / normDcR
    val dwR  = nDwR[i].toDouble()  / normDwR
    val rr   = if (normRR > 0L) nRR[i].toDouble() / normRR else 0.0
    val xi   = if (rr > 0.0) (dcDw - dcR - dwR + rr) / rr else Double.NaN
    val err  = if (nRR[i] > 0L) (1.0 + xi) / sqrt(nRR[i].toDouble()) else Double.NaN
    XiBin(centers[i], xi, err, nRR[i])
}
println("ξ_CW 計算完了  (nBins = $nBins)")

ξ_CW 計算完了  (nBins = 38)


In [7]:
// --- 広域プロット: ξ_CC vs ξ_CW (xi_cc.csv から読み込み) ---
val xiCcDf2 = DataFrame.readCSV("./output/xi/xi_cc.csv")
val rCC   = xiCcDf2.rows().map { it["r_km"] as Double }
val xiCC_ = xiCcDf2.rows().map { it["xi"]   as Double }

val rLong  = rCC + xiCW.map { it.rCenter }
val xiLong = xiCC_ + xiCW.map { it.xi }
val lbl    = List(rCC.size) { "ξ_CC  Coles" } + List(xiCW.size) { "ξ_CW  Cross" }

letsPlot(mapOf("r" to rLong, "xi" to xiLong, "label" to lbl)) +
    geomLine(size = 1.2)  { x = "r"; y = "xi"; color = "label" } +
    geomPoint(size = 1.8) { x = "r"; y = "xi"; color = "label" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "r [km]") + scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf("ξ_CC  Coles" to "#4682B4", "ξ_CW  Cross" to "#E87040")) +
    ggtitle("ξ_CC(r) vs ξ_CW(r)", "点線: r_D=71.3 km | 赤点線: BAO 666 km") +
    ggsize(800, 450)

<path d="M0.0 40.79138062295203 L0.0 40.79138062295203 L15.905223864781362 16.000000000000057 L31.810447729562753 59.339574342961384 L47.715671594344116 98.22507886100072 L63.62089545912548 105.02798292130765 L79.5261193239069 126.13265438402246 L95.43134318868826 154.65450999208824 L111.33656705346962 182.1737338657643 L127.24179091825098 208.4571790742472 L143.1470147830324 238.16025148811718 L159.05223864781377 264.517222911585 L174.95746251259519 281.4857318625333 L190.86268637737655 292.87006297587004 L206.7679102421579 304.2707406090255 L222.67313410693927 313.72687793414013 L238.57835797172064 321.2345278076001 L254.483581836502 325.4719536746652 L270.38880570128345 328.6234407069664 L286.29402956606475 328.61994534176176 L302.19925343084617 330.7171831176222 L318.1044772956275 331.80188439210264 L334.0097011604089 333.3155308982776 L349.9149250251902 333.79319321904904 L365.82014888997173 332.96627798455575 L381.72537275475304 334.41987610807604 L397.63059661953434 334.76765301446795 L413.53582048431576 334.60406415633935 L429.44104434909707 334.5194096482576 L445.3462682138785 334.8113723043194 L461.25149207866 334.4511159381578 L477.1567159434413 331.2391070753537 L493.0619398082225 333.4493919658451 L508.96716367300405 335.023432679581 L524.8723875377855 335.3350567563924 L540.7776114025668 335.06395335839767 L556.6828352673482 334.6742859751648 L572.5880591321296 335.5416921929067 L588.4932829969109 335.99569191384614 " fill="none" stroke-width="2.64" stroke="rgb(70,130,180)" stroke-opacity="1.0">
 
 
 
 <path d="M0.0 62.53083954437852 L0.0 62.53083954437852 L15.905223864781362 61.08047880425147 L31.810447729562753 77.9145835879599 L47.715671594344116 113.92035910478464 L63.62089545912548 126.46338637459874 L79.5261193239069 136.47668578291874 L95.43134318868826 166.71887812401266 L111.33656705346962 194.52671339907752 L127.24179091825098 218.92565889892 L143.1470147830324 245.43882496510383 L159.05223864781377 268.3203885628647 L174.95746251259519 285.84659724952 L190.86268637737655 297.95045825450876 L206.7679102421579 307.7268938686778 L222.67313410693927 315.07628885417364 L238.57835797172064 320.5392377423627 L254.483581836502 324.9344970980136 L270.38880570128345 328.15079007936043 L286.29402956606475 328.661366004516 L302.19925343084617 330.3211576189949 L318.1044772956275 331.67926443079375 L334.0097011604089 333.08988507606074 L349.9149250251902 333.57281105702657 L365.82014888997173 332.92421681272896 L381.72537275475304 334.2673284357828 L397.63059661953434 334.5664778028238 L413.53582048431576 334.51238136628956 L429.44104434909707 334.3672139036957 L445.3462682138785 334.7434419390191 L461.25149207866 334.2973360668615 L477.1567159434413 331.41483763511957 L493.0619398082225 333.444971945351 L508.96716367300405 334.9483891359053 L524.8723875377855 335.2279817461462 L540.7776114025668 335.0600631144922 L556.6828352673482 334.6574172351891 L572.5880591321296 335.49125727480356 L588.4932829969109 336.0 " fill="none" stroke-width="2.64" stroke="rgb(232,112,64)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 250 
 
 
 
 
 
 
 
 
 ξ_CC(r) vs ξ_CW(r) 
 
 
 
 
 点線: r_D=71.3 km | 赤点線: BAO 666 km 
 
 
 
 
 ξ(r) 
 
 
 
 
 r [km] 
 
 
 
 
 
 
 
 
 label 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ξ_CC Coles 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ξ_CW Cross

In [8]:
// --- 小スケール線形ビン (Δr = 1 km, r = 0.5–30.5 km) ---
val binsSmall = DoubleArray(31) { i -> 0.5 + i.toDouble() }   // [0.5, 1.5, ..., 30.5]
val centersSmall = DoubleArray(30) { i -> 1.0 + i.toDouble() }

println("D_C × D_W (小スケール) を計算中...")
val nDcDwSm = pairCounts(colesPoints, woolPoints, binsSmall)
println("D_C × R (小スケール) を計算中...")
val nDcRSm  = pairCounts(colesPoints, randPoints, binsSmall)
println("D_W × R (小スケール) を計算中...")
val nDwRSm  = pairCounts(woolPoints,  randPoints, binsSmall)
println("R × R (小スケール) を計算中...")
val nRRSm   = pairCounts(randPoints, null, binsSmall)

val xiCwSmall: List<XiBin> = centersSmall.indices.map { i ->
    val dcDw = nDcDwSm[i].toDouble() / normDcDw
    val dcR  = nDcRSm[i].toDouble()  / normDcR
    val dwR  = nDwRSm[i].toDouble()  / normDwR
    val rr   = if (normRR > 0L) nRRSm[i].toDouble() / normRR else 0.0
    val xi   = if (rr > 0.0) (dcDw - dcR - dwR + rr) / rr else Double.NaN
    val err  = if (nRRSm[i] > 0L) (1.0 + xi) / sqrt(nRRSm[i].toDouble()) else Double.NaN
    XiBin(centersSmall[i], xi, err, nRRSm[i])
}
println("小スケール ξ_CW 完了")

D_C × D_W (小スケール) を計算中...
D_C × R (小スケール) を計算中...
D_W × R (小スケール) を計算中...
R × R (小スケール) を計算中...
小スケール ξ_CW 完了


In [9]:
// --- CSV 出力 (wide + small) ---
java.io.File("./output/xi/xi_cw_wide.csv").bufferedWriter().use { w ->
    w.appendLine("r_km,xi,xi_err,n_rr")
    xiCW.forEach { b -> w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}") }
}
println("保存: output/xi/xi_cw_wide.csv  ($nBins ビン)")

java.io.File("./output/xi/xi_cw_small.csv").bufferedWriter().use { w ->
    w.appendLine("r_km,xi,xi_err,n_rr")
    xiCwSmall.forEach { b -> w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}") }
}
println("保存: output/xi/xi_cw_small.csv  (30 ビン)")

保存: output/xi/xi_cw_wide.csv  (38 ビン)
保存: output/xi/xi_cw_small.csv  (30 ビン)
